In [2]:
# Libraries
from matplotlib import pyplot as plt
import numpy as np
from sort import Sort, parse_args
import os
import glob
import time
import cv2
import motmetrics as mm
import pandas as pd

In [3]:
# Variables
phase = 'train'
data_path = 'MOT20'
total_time = 0.0
total_frames = 0
colours = np.random.rand(32, 3)*255 #used only for display

pattern = os.path.join(data_path, phase, '*', 'det', 'det.txt')
colnames=['frame', 'id', 'left_x', 'left_y', 'right_x', 'right_y', 'a', 'b', 'c']

In [4]:
def create_video(sequence, output_path, fps=24):
    # Get the height and width of the frames
    height, width = sequence[0].shape[:2]

    # Define the codec and create a VideoWriter object
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    video_writer = cv2.VideoWriter(output_path, fourcc, fps, (width, height), isColor=True)

    # Iterate through the frames in each list
    for frame in sequence:
        
        # Write the frame to the video file
        video_writer.write(frame)

    # Release the VideoWriter object
    video_writer.release()


In [5]:
# Create an accumulator that will be updated during each frame
acc = mm.MOTAccumulator(auto_id=True)

for seq_dets_fn in glob.glob(pattern):
    acc_seq = mm.MOTAccumulator(auto_id=True)

    sequence = []
    mot_tracker = Sort(max_age=1, 
                       min_hits=3,
                       iou_threshold=0.3) #create instance of the SORT tracker

    seq_dets = np.loadtxt(seq_dets_fn, delimiter=',')
    seq = seq_dets_fn[pattern.find('*'):].split(os.path.sep)[0]
    boxes_coords = pd.read_csv('./MOT20/train/'+seq+'/gt/gt.txt', sep=',', names=colnames)
    
    for frame in range(int(seq_dets[:,0].max())):

        # Object detection
        frame += 1 
        dets = seq_dets[seq_dets[:, 0]==frame, 2:7]
        dets[:, 2:4] += dets[:, 0:2] #convert to [x1,y1,w,h] to [x1,y1,x2,y2]
        total_frames += 1

        # Objects estimation
        start_time = time.time()
        trackers = mot_tracker.update(dets)
        cycle_time = time.time() - start_time
        total_time += cycle_time

        # Reading frame
        fn = os.path.join(data_path, phase, seq, 'img1', '%06d.jpg'%(frame))
        im = cv2.imread(fn)

        # Ground truth objects
        boxes_frame = boxes_coords[boxes_coords['frame'] == frame]
        obj_coord = boxes_frame[['left_x','left_y','right_y','right_y']].values

        trackers_id = []
        hyp_coord = np.zeros((len(trackers), 4))

        for i, d in enumerate(trackers):
            # Hypothesis objects
            trackers_id.append(i)
            d_coord = [d[0], d[1], d[2] - d[0], d[3] - d[1]]
            hyp_coord[i] = d_coord

            # Drawing rectangles
            d = d.astype(np.int32)
            cv2.rectangle(im, (int(d[0]), int(d[1])), (int(d[2]), int(d[3])), colours[int(d[4]%32),:], 6)

        sequence.append(im)

        # Calculate distances for updating the acumulator
        distance_matrix = mm.distances.iou_matrix(obj_coord, hyp_coord, max_iou=0.7)

        # Call update once for per frame - General
        acc.update(
            boxes_frame['id'],          # Ground truth object ID in this frame
            trackers_id,                # Detector hypothesis ID in this frame
            [distance_matrix])
        
        # Call update once for per frame - For sequence
        acc_seq.update(
            boxes_frame['id'],          # Ground truth object ID in this frame
            trackers_id,                # Detector hypothesis ID in this frame
            [distance_matrix])

    print(f"Total Tracking took for {seq}: %.3f seconds for %d frames or %.1f FPS" % (total_time, total_frames, total_frames / total_time))
    
    mh = mm.metrics.create()

    summary = mh.compute(
        acc_seq,
        metrics=mm.metrics.motchallenge_metrics
        )


    strsummary = mm.io.render_summary(
        summary,
        formatters=mh.formatters,
        namemap=mm.io.motchallenge_metric_names
    )
    print(strsummary)

    create_video(sequence, './output_'+seq+'.mp4', fps=24)

Total Tracking took for MOT20-01: 3.628 seconds for 429 frames or 118.3 FPS
  IDF1   IDP  IDR  Rcll  Prcn GT MT PT ML  FP    FN  IDs   FM  MOTA  MOTP  IDt IDa IDm
0 7.3% 11.9% 5.3% 43.4% 97.4% 90 14 51 25 309 15089 4798  759 24.2% 0.563 4649   5  47
Total Tracking took for MOT20-02: 29.273 seconds for 3211 frames or 109.7 FPS
  IDF1  IDP  IDR  Rcll  Prcn  GT MT  PT ML   FP     FN   IDs    FM  MOTA  MOTP   IDt IDa IDm
0 2.2% 3.7% 1.6% 41.2% 98.2% 296 48 199 49 1521 118922 36270  4916 22.5% 0.570 35299  20 253
Total Tracking took for MOT20-03: 70.826 seconds for 5616 frames or 79.3 FPS
  IDF1  IDP  IDR  Rcll  Prcn  GT MT  PT  ML   FP     FN    IDs     FM MOTA  MOTP    IDt IDa IDm
0 1.1% 1.9% 0.8% 41.7% 97.3% 735 52 472 211 4084 208126 116082  12512 8.0% 0.551 115499  25 599
Total Tracking took for MOT20-05: 152.389 seconds for 8931 frames or 58.6 FPS
  IDF1  IDP  IDR  Rcll  Prcn   GT MT  PT  ML   FP     FN    IDs     FM MOTA  MOTP    IDt IDa  IDm
0 1.0% 1.7% 0.8% 43.7% 97.5% 1211 73 870 

In [6]:
# Metrics for all the sequences
mh = mm.metrics.create()

summary = mh.compute(
    acc,
    metrics=mm.metrics.motchallenge_metrics
    )


strsummary = mm.io.render_summary(
    summary,
    formatters=mh.formatters,
    namemap=mm.io.motchallenge_metric_names
)
print(strsummary)

  IDF1  IDP  IDR  Rcll  Prcn   GT MT  PT  ML    FP     FN    IDs     FM MOTA  MOTP    IDt IDa  IDm
0 0.9% 1.5% 0.7% 42.7% 97.6% 1211 44 975 192 14202 765396 424789  48552 9.9% 0.559 421190  85 1133
